# FCT RQ1 — Greedy Set Cover for Minimum-Cost Test Subset

In [41]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

In [ ]:
DATA_PATH = "C:\\FCT_EDA\\clean_merged_dataset.csv"

df = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Raw shape: {df.shape}")

In [ ]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace(r'[^a-z0-9_]', '', regex=True)
)
print(f"Columns: {df.columns.tolist()}")

In [ ]:
df = df[~df['no'].astype(str).str.lower().isin(['no', 'nan'])]
df = df[~df['step_name_1'].astype(str).str.lower().str.contains('step name', na=False)]
df.reset_index(drop=True, inplace=True)
print(f"After cleaning headers: {df.shape}")

In [ ]:
# Each source filename is a unique board run identifier
df['unit_id'] = df['source_file'].astype(str).str.strip()
print(f"Unique units: {df['unit_id'].nunique()}")

In [ ]:
df['result'] = df['result'].astype(str).str.strip().str.upper()
df['outcome_from_filename'] = df['outcome_from_filename'].astype(str).str.strip().str.upper()

# Conservative encoding: PASS=1, FAIL/ABORT=0
# ABORT treated as potential quality signal
df['result_binary'] = df['result'].map({'PASS': 1, 'FAIL': 0, 'ABORT': 0})
df['unit_pass'] = df['outcome_from_filename'].map({'PASS': 1, 'FAIL': 0, 'ABORT': 0})

print(f"Result values: {df['result'].unique()}")
print(f"Unit outcomes: {df['outcome_from_filename'].unique()}")

In [ ]:
df['step_time'] = pd.to_numeric(df['step_time'], errors='coerce').fillna(0.0)
df['no'] = pd.to_numeric(df['no'], errors='coerce')
df['step_name_1'] = df['step_name_1'].astype(str).str.strip()
df['step_name_2'] = df['step_name_2'].astype(str).str.strip()

# Construct unique test step identifier from 
# two hierarchical name fields
df['test_id'] = df['step_name_1'] + " / " + df['step_name_2']

sample = df[['step_name_1', 'step_name_2', 'test_id']].drop_duplicates().head(20)
print(sample.to_string(index=False))

In [ ]:
# Aggregate step-level data to unit level
unit_summary = df.groupby('unit_id').agg(
    overall_outcome=('unit_pass', 'first'),
    total_test_time=('step_time', 'sum'),
    num_steps_executed=('test_id', 'count'),
    num_steps_failed=('result_binary', lambda x: (x == 0).sum())
).reset_index()

n_total   = len(unit_summary)
n_pass    = unit_summary['overall_outcome'].sum()
n_fail    = (unit_summary['overall_outcome'] == 0).sum()
dpm       = round(n_fail / n_total * 1_000_000, 1)
print(f"\n{'='*50}")
print("UNIT SUMMARY")
print(f"{'='*50}")
print(f"Total units:        {n_total}")
print(f"Passing units:      {n_pass}")
print(f"Failing units:      {n_fail}")
print(f"DPM (observed):     {dpm}")
print(f"Mean test time:     {unit_summary['total_test_time'].mean():.1f} s")
print(f"Min  test time:     {unit_summary['total_test_time'].min():.1f} s")
print(f"Max  test time:     {unit_summary['total_test_time'].max():.1f} s")

In [ ]:
all_failing = unit_summary[
    unit_summary['overall_outcome'] == 0
]['unit_id'].tolist()

genuine_fail_units = []
abort_only_units   = []

# Separate genuine defects from equipment-induced aborts
# A unit is genuine fail only if at least one step = FAIL
for unit_id in all_failing:
    unit_steps    = df[df['unit_id'] == unit_id]
    has_step_fail = (unit_steps['result'] == 'FAIL').any()
    if has_step_fail:
        genuine_fail_units.append(unit_id)
    else:
        abort_only_units.append(unit_id)

failing_units = genuine_fail_units
N_fail        = len(failing_units)

print(f"Total failing:    {len(all_failing)}")
print(f"Genuine FAIL:     {len(genuine_fail_units)}")
print(f"ABORT-only:       {len(abort_only_units)}")
print(f"Using:            {N_fail} units")

In [ ]:
# Compute per-step diagnostic statistics
# Identifies which steps detect most defects
step_summary = df.groupby('test_id').agg(
    mean_cost_s=('step_time', 'mean'),
    total_executions=('result_binary', 'count'),
    total_fails=('result_binary', lambda x: (x == 0).sum()),
    fail_rate_pct=('result_binary', lambda x: round((x == 0).mean() * 100, 4))
).reset_index().sort_values('total_fails', ascending=False)

print(f"\n{'='*50}")
print("TEST STEP SUMMARY")
print(f"{'='*50}")
print(f"Total unique test steps:     {len(step_summary)}")
print(f"Steps with at least 1 FAIL:  {(step_summary['total_fails'] > 0).sum()}")
print(f"Steps with zero FAILs:       {(step_summary['total_fails'] == 0).sum()}")
print(f"\nTop 15 most failing steps:")
print(step_summary.head(15)[[
    'test_id','mean_cost_s','total_executions','total_fails','fail_rate_pct'
]].to_string(index=False))

In [ ]:
# Load preprocessed outcome matrix and cost vector
outcome_matrix = pd.read_csv('fct_outcome_matrix.csv', index_col=0)
print(f"Outcome matrix: {outcome_matrix.shape}")

cost_vector = pd.read_csv('fct_cost_vector.csv', index_col=0)
cost_vector.columns = ['mean_cost_s']
print(f"Cost vector: {len(cost_vector)} rows")
print(f"C_full cost: {cost_vector['mean_cost_s'].sum():.2f}s")

In [ ]:
# Check for ESPtool capitalisation variants
# Two duplicate columns found and merged in preprocessing
esp_cols = [c for c in outcome_matrix.columns 
            if 'ESPtool' in c 
            or 'espTool' in c.lower() 
            or 'Esptools' in c.lower()]
print("ESPtool columns found:")
for c in esp_cols:
    print(f"  '{c}'")

print(f"\nTotal columns: {outcome_matrix.shape[1]}")

In [ ]:
# Remove duplicate ESPtool capitalisation variants
# from cost vector to match 172-column outcome matrix
cost_vector = pd.read_csv("fct_cost_vector.csv", 
                           index_col=0, header=0)
cost_vector.columns = ['mean_cost_s']
print(f"Before: {len(cost_vector)} rows")

col2 = '(FIT) Check ID instance for ESPtool / Powershell USB'
col4 = '(FIT) Found COM for ESPtool / Powershell USB'

cost_vector = cost_vector.drop(
    index=[col2, col4], errors='ignore'
)
print(f"After:  {len(cost_vector)} rows")

cost_vector.to_csv("fct_cost_vector.csv", header=True)
print("Saved fct_cost_vector.csv")

In [ ]:
fail_matrix = outcome_matrix.loc[
    outcome_matrix.index.isin(failing_units)
].copy()

# Step detects a unit if result = 0 (FAIL or ABORT)
detection_matrix = (fail_matrix == 0).astype(float).fillna(0)

print(f"\nDetection matrix: {detection_matrix.shape}")

In [ ]:
step_detection_count = detection_matrix.sum(axis=0)
diagnostic_steps     = step_detection_count[step_detection_count > 0].index.tolist()
non_diagnostic_steps = step_detection_count[step_detection_count == 0].index.tolist()

print(f"\nDiagnostic steps:     {len(diagnostic_steps)}")
print(f"Non-diagnostic steps: {len(non_diagnostic_steps)}")

In [ ]:
# Build detection sets for all diagnostic steps
detection_sets = {}
for step in diagnostic_steps:
    detection_sets[step] = set(
        detection_matrix.index[
            detection_matrix[step] == 1
        ].tolist()
    )
print(f"Detection sets built: {len(detection_sets)} steps")

In [ ]:
print(f"\n{'='*55}")
print("GREEDY SET COVER — selecting C*")
print(f"{'='*55}")

costs_full = cost_vector.reindex(
    outcome_matrix.columns
)['mean_cost_s'].fillna(0)

# Greedy set cover — iteratively selects the step
# with highest ratio of newly covered units to cost
already_used = set()
selected     = []
trace        = []

uncovered = set(failing_units)

print(f"Total units to cover: {len(uncovered)}")
print(f"{'─'*55}")

step_num = 1
while uncovered:
    best_step  = None
    best_score = -1
    best_newly = set()

    for step in diagnostic_steps:
        if step in already_used:
            continue
        newly_detected = detection_sets.get(
            step, set()
        ) & uncovered
        if not newly_detected:
            continue
        cost_eff = max(costs_full.get(step, 0.001), 0.001)
        score    = len(newly_detected) / cost_eff
        if score > best_score:
            best_score = score
            best_step  = step
            best_newly = newly_detected

    if best_step is None:
        break

    already_used.add(best_step)
    uncovered -= best_newly
    selected.append(best_step)
    trace.append({
        'rank':                 step_num,
        'test_id':              best_step,
        'cost_s':               round(costs_full.get(best_step, 0), 4),
        'units_newly_detected': len(best_newly),
        'units_remaining':      len(uncovered),
        'cumulative_cost_s':    round(
            sum(costs_full.get(s, 0) for s in selected), 2
        )
    })

    print(f"{step_num:2d}. {best_step[:55]:<55} "
          f"detects {len(best_newly):3d} | "
          f"remaining {len(uncovered):3d}")
    step_num += 1

print(f"\nC* total steps:  {len(selected)}")
print(f"C* total cost:   "
      f"{sum(costs_full.get(s,0) for s in selected):.4f}s")
print(f"Escaped units:   {len(uncovered)}")

In [ ]:
      c_star_df = pd.DataFrame({'test_id': selected})
      c_star_df.to_csv('fct_rq1_c_star.csv', index=False)
      print(f"Saved: fct_rq1_c_star.csv — {len(selected)} steps")

      C_star_cost = sum(costs_full.get(s, 0) for s in selected)

      cost_saving_s    = C_full_cost - C_star_cost
      cost_saving_pct  = cost_saving_s / C_full_cost * 100
      steps_saving_pct = (1 - len(selected) / outcome_matrix.shape[1]) * 100
      escaped          = len(uncovered)
      escape_risk      = escaped / len(failing_units)
      escape_dpm       = escape_risk * 1_000_000

      print(f"\n{'='*55}")
      print("RQ1 RESULTS — MINIMUM COST SUBSET C*")
      print(f"{'='*55}")
      print(f"Steps in C*:               {len(selected)} / {outcome_matrix.shape[1]}")
      print(f"Steps eliminated:          "
            f"{outcome_matrix.shape[1]-len(selected)} "
            f"({steps_saving_pct:.1f}%)")
      print(f"Cost of C* per unit:       {C_star_cost:.2f} seconds")
      print(f"Cost of full sequence:     {C_full_cost:.2f} seconds")
      print(f"Time saved per unit:       {cost_saving_s:.2f} seconds")
      print(f"Cost saving:               {cost_saving_pct:.1f}%")
      print(f"Failing units detected:    "
            f"{len(failing_units)-escaped} / {len(failing_units)}")
      print(f"Escaped units:             {escaped}")
      print(f"Escape risk:               {escape_risk:.6f}")
      print(f"Escape DPM:                {escape_dpm:.1f}")

In [ ]:
# Save reduced test plan C_red — used in RQ2 and RQ3
pd.DataFrame({'test_id': selected}).to_csv(
    'fct_rq2_cred.csv', index=False
)

C_red      = selected
C_red_cost = sum(
    float(cost_vector.loc[s, 'mean_cost_s'])
    if s in cost_vector.index else 0.0
    for s in C_red
)

print(f"Saved fct_rq2_cred.csv")
print(f"Steps:        {len(C_red)}")
print(f"C_full cost:  {C_full_cost:.4f}s")
print(f"C_red cost:   {C_red_cost:.4f}s")
print(f"Saving:       {C_full_cost - C_red_cost:.4f}s")
print(f"Saving %:     {(C_full_cost - C_red_cost)/C_full_cost*100:.2f}%")